# Data Exploration

This notebook explores the two main data sources:
- **POINTDATA** — PCR-GLOBWB 2 model output: gridded NetCDF files (lat/lon/time) over the contiguous US
- **ResOpsUS+CARS_v10** — observed US reservoir operations with static attributes, time series (CSV + NetCDF4), and GIS shapefiles

In [2]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd

warnings.filterwarnings('ignore', category=FutureWarning)

POINTDATA_DIR = 'Data/POINTDATA'
RESOPS_DIR    = 'Data/ResOpsUS+CARS_v10/v1.0'
PARAM_DIR     = os.path.join(POINTDATA_DIR, '10_param_RF_bounds_final')

---
## 1  POINTDATA — Gridded PCR-GLOBWB 2 output

All files share the same spatial grid: **312 × 348 cells** (~0.083° ≈ 5 arcmin resolution) covering the contiguous US (lat 18–44 °N, lon −125 to −96 °E).

Three temporal resolutions:
- **daily** (16 436 steps, 1979-01-01 → 2023-12-31)
- **monthly** (540 steps)
- **annual** (45 steps)

In [3]:
# --- Inventory all NetCDF files in POINTDATA (top-level only) ---
nc_files = sorted(glob.glob(os.path.join(POINTDATA_DIR, '*.nc')))

rows = []
for path in nc_files:
    fname = os.path.basename(path)
    try:
        ds = xr.open_dataset(path, engine='scipy')
        var_name = list(ds.data_vars)[0]
        t_len    = ds.sizes.get('time', None)
        t_start  = str(ds.time.values[0])[:10] if 'time' in ds.coords else None
        t_end    = str(ds.time.values[-1])[:10] if 'time' in ds.coords else None
        size_mb  = os.path.getsize(path) / 1e6
        rows.append(dict(file=fname, variable=var_name,
                         time_steps=t_len, start=t_start, end=t_end,
                         size_mb=round(size_mb, 1)))
        ds.close()
    except Exception:
        rows.append(dict(file=fname, variable='[NetCDF4 — needs netcdf4 lib]',
                         time_steps=None, start=None, end=None,
                         size_mb=round(os.path.getsize(path) / 1e6, 1)))

inv = pd.DataFrame(rows)
print(f"Total files: {len(inv)}")
inv

Total files: 49


,file,variable,time_steps,start,end,size_mb
0,RensReduction_monthAvg_output.nc,RensReduction_factor,540.0,1979-01-31,2023-12-31,234.5
1,actualET_annuaTot_output.nc,land_surface_evaporation,45.0,1979-12-31,2023-12-31,19.5
2,desalinationAbstraction_monthTot_output.nc,desalination_source_abstraction,540.0,1979-01-31,2023-12-31,234.5
3,discharge_annuaAvg_output.nc,discharge,45.0,1979-12-31,2023-12-31,19.5
4,discharge_dailyTot_output.nc,discharge,16436.0,1979-01-01,2023-12-31,7138.3
5,domesticWaterWithdrawal_monthTot_output.nc,domestic_water_withdrawal,540.0,1979-01-31,2023-12-31,234.5
6,fossilGroundwaterAbstraction_monthTot_output.nc,fossil_groundwater_abstraction,540.0,1979-01-31,2023-12-31,234.5
7,gwRecharge_annuaTot_output.nc,groundwater_recharge,45.0,1979-12-31,2023-12-31,19.5
8,industryWaterWithdrawal_monthTot_output.nc,industry_water_withdrawal,540.0,1979-01-31,2023-12-31,234.5
9,irrNonPaddyWaterWithdrawal_monthTot_output.nc,non_paddy_irrigation_withdrawal,540.0,1979-01-31,2023-12-31,234.5


In [4]:
# --- Temporal resolution breakdown ---
daily   = inv[inv.time_steps == 16436]
monthly = inv[inv.time_steps == 540]
annual  = inv[inv.time_steps == 45]
other   = inv[~inv.time_steps.isin([16436, 540, 45])]

print(f"Daily files   (16 436 steps, 1979–2023): {len(daily)}")
print(f"Monthly files (540 steps):                {len(monthly)}")
print(f"Annual files  ( 45 steps):                {len(annual)}")
print(f"Other / NetCDF4:                          {len(other)}")

Daily files   (16 436 steps, 1979–2023): 5
Monthly files (540 steps):                24
Annual files  ( 45 steps):                18
Other / NetCDF4:                          2


In [5]:
# --- Inspect one file in detail: daily discharge ---
ds_q = xr.open_dataset(
    os.path.join(POINTDATA_DIR, 'discharge_dailyTot_output.nc'), engine='scipy'
)
print(ds_q)
print("\nLat:", float(ds_q.lat.min()), "→", float(ds_q.lat.max()))
print("Lon:", float(ds_q.lon.min()), "→", float(ds_q.lon.max()))
print("Time:", str(ds_q.time.values[0])[:10], "→", str(ds_q.time.values[-1])[:10])

<xarray.Dataset> Size: 7GB
Dimensions:    (time: 16436, lat: 312, lon: 348)
Coordinates:
  * time       (time) datetime64[ns] 131kB 1979-01-01 1979-01-02 ... 2023-12-31
  * lat        (lat) float32 1kB 43.96 43.88 43.79 43.71 ... 18.21 18.12 18.04
  * lon        (lon) float32 1kB -125.0 -124.9 -124.8 ... -96.21 -96.12 -96.04
Data variables:
    discharge  (time, lat, lon) float32 7GB ...
Attributes:
    institution:  Department of Physical Geography, Utrecht University
    title:        PCR-GLOBWB 2 output for SOS Water
    description:  by Jennie C. Steyaert (contact: j.c.steyaert@uu.nl)

Lat: 18.04166603088379 → 43.95833206176758
Lon: -124.95833587646484 → -96.04166412353516
Time: 1979-01-01 → 2023-12-31


In [6]:
# --- Spatial map: mean daily discharge ---
mean_q = ds_q['discharge'].mean('time').compute()

fig, ax = plt.subplots(figsize=(10, 5))
img = ax.pcolormesh(
    ds_q.lon.values, ds_q.lat.values, mean_q.values,
    norm=mcolors.LogNorm(vmin=0.1, vmax=float(mean_q.values[mean_q.values > 0].max())),
    cmap='Blues'
)
plt.colorbar(img, ax=ax, label='Mean discharge (m³/s, log scale)')
ax.set_title('POINTDATA — Mean Daily Discharge (1979–2023)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()
ds_q.close()

KeyboardInterrupt: 

In [ ]:
# --- Spatial map: mean storage fraction (monthly) ---
ds_sf = xr.open_dataset(
    os.path.join(POINTDATA_DIR, 'sos_stor_fraction_monthEnd_output.nc'), engine='scipy'
)
mean_sf = ds_sf['storage fraction'].mean('time').compute()

fig, ax = plt.subplots(figsize=(10, 5))
img = ax.pcolormesh(
    ds_sf.lon.values, ds_sf.lat.values, mean_sf.values,
    vmin=0, vmax=1, cmap='RdYlBu'
)
plt.colorbar(img, ax=ax, label='Mean storage fraction (0–1)')
ax.set_title('POINTDATA — Mean Monthly Reservoir Storage Fraction')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()
ds_sf.close()

### 1.1  `10_param_RF_bounds_final` subfolder

One NetCDF4 file per simulation date; each holds RF-predicted reservoir operation parameters.  
Requires the `netcdf4` library (`pip install netcdf4`). The cell below lists the file count, date range, and inspects one file if the library is available.

In [ ]:
param_files = sorted(glob.glob(os.path.join(PARAM_DIR, '*.nc')))
print(f"Number of parameter NC files: {len(param_files)}")

dates = []
for f in param_files:
    stem = os.path.splitext(os.path.basename(f))[0]
    try:
        dates.append(pd.to_datetime(stem))
    except Exception:
        pass
dates = sorted(dates)
print(f"Date range:  {dates[0].date()} → {dates[-1].date()}")
print(f"\nSample files:")
for f in param_files[:5]:
    print(f"  {os.path.basename(f):20s}  {os.path.getsize(f)/1e3:.1f} KB")

In [ ]:
try:
    import netCDF4 as nc4
    with nc4.Dataset(param_files[0]) as ds:
        print("Variables:", list(ds.variables.keys()))
        for v in ds.variables:
            var = ds.variables[v]
            print(f"  {v}: shape={var.shape}, dtype={var.dtype}")
except ImportError:
    print("netcdf4 not installed.  Run:  pip install netcdf4")
    print("Files are NetCDF4; each named YYYY-M-D.nc corresponds to one model timestep.")

---
## 2  ResOpsUS+CARS — Observed US Reservoir Operations

**676 reservoirs** across the contiguous US (Steyaert et al. 2022 + ERA5/GloFAS forcings).

```
ResOpsUS+CARS_v10/v1.0/
├── attributes/   ← 5 CSV files: GRanD, ResOps metadata, climate indices, GloFAS params & static maps
├── GIS/          ← Shapefiles: 677 reservoir points + catchment polygons
└── time_series/
    ├── csv/      ← 676 CSVs, one per reservoir (GRAND_ID.csv)
    └── netcdf/   ← same 676 series as NetCDF4
```

In [ ]:
# --- Attribute files overview ---
attr_dir   = os.path.join(RESOPS_DIR, 'attributes')
attr_files = sorted(glob.glob(os.path.join(attr_dir, '*.csv')))

for path in attr_files:
    df = pd.read_csv(path)
    print(f"{os.path.basename(path):40s}  rows={len(df):4d}  cols={df.shape[1]:3d}")
    print(f"  Columns: {list(df.columns[:8])}{'...' if df.shape[1] > 8 else ''}")

In [ ]:
# --- Load attribute tables ---
grand   = pd.read_csv(os.path.join(attr_dir, 'grand.csv'))
resops  = pd.read_csv(os.path.join(attr_dir, 'resops.csv'))
glofas  = pd.read_csv(os.path.join(attr_dir, 'glofas.csv'))
climate = pd.read_csv(os.path.join(attr_dir, 'climate_indices.csv'))

print("GRanD attributes — shape:", grand.shape)
grand.head(3)

In [ ]:
print("ResOps metadata — shape:", resops.shape)
resops.head(3)

In [ ]:
# --- GRanD: reservoir characteristic distributions ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

grand['CAP_MCM'].dropna().plot.hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Capacity (MCM)')
axes[0].set_title('Storage capacity')

grand['DAM_HGT_M'].dropna().plot.hist(bins=40, ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_xlabel('Height (m)')
axes[1].set_title('Dam height')

grand['CATCH_SKM'].dropna().plot.hist(bins=40, ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_xlabel('Area (km²)')
axes[2].set_title('Catchment area')

for ax in axes:
    ax.set_ylabel('Count')
plt.suptitle('GRanD — Reservoir characteristics (n=676)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Primary use breakdown ---
use_cols   = [c for c in grand.columns if c.startswith('USE_')]
use_counts = (grand[use_cols] > 0).sum().sort_values(ascending=False)
use_counts.index = [c.replace('USE_', '') for c in use_counts.index]

fig, ax = plt.subplots(figsize=(8, 4))
use_counts.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Reservoirs by use type (GRanD)')
ax.set_ylabel('Number of reservoirs')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Spatial distribution of reservoirs and catchments ---
try:
    gdf_res   = gpd.read_file(os.path.join(RESOPS_DIR, 'GIS', 'reservoirs_677_3sec.shp'))
    gdf_catch = gpd.read_file(os.path.join(RESOPS_DIR, 'GIS', 'catchments_3sec.shp'))

    fig, ax = plt.subplots(figsize=(12, 7))
    gdf_catch.plot(ax=ax, color='lightgrey', edgecolor='grey', linewidth=0.3, alpha=0.5)
    gdf_res.plot(ax=ax, markersize=6, color='steelblue', alpha=0.7)
    ax.set_title('ResOpsUS — Reservoir locations and catchments (n=677)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()
    print("Reservoir GDF columns:", list(gdf_res.columns))
    gdf_res.head(3)
except Exception as e:
    print("GIS plot failed:", e)

In [ ]:
# --- Time-series CSV: column overview ---
ts_dir   = os.path.join(RESOPS_DIR, 'time_series', 'csv')
ts_files = sorted(glob.glob(os.path.join(ts_dir, '*.csv')))
print(f"Number of time-series CSV files: {len(ts_files)}")

example = pd.read_csv(ts_files[0], parse_dates=['date'])
print(f"\nExample reservoir GRAND_ID={os.path.splitext(os.path.basename(ts_files[0]))[0]}")
print(f"  Date range: {example.date.min().date()} → {example.date.max().date()}")
print(f"  Shape:      {example.shape}")
print("\nColumn overview:")
for col in example.columns:
    n_nan = int(example[col].isna().sum())
    print(f"  {col:35s}  NaN={n_nan:5d} / {len(example)}")

In [ ]:
# --- Data availability: how many reservoirs have each key variable? ---
key_cols = ['storage', 'outflow', 'elevation', 'inflow_sim',
            'evapo_areal', 'temp_areal', 'precip_areal']
availability = {col: 0 for col in key_cols}

for path in ts_files:
    df = pd.read_csv(path, usecols=lambda c: c in key_cols + ['date'])
    for col in key_cols:
        if col in df.columns and df[col].notna().any():
            availability[col] += 1

avail_df = pd.Series(availability, name='reservoirs_with_data').sort_values(ascending=False)
print("Variable availability across 676 reservoirs:")
print(avail_df.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
avail_df.plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.axvline(len(ts_files), color='red', linestyle='--', label='Total (676)')
ax.set_xlabel('Number of reservoirs')
ax.set_title('ResOpsUS — Data availability by variable')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Record length distribution (non-NaN storage days) ---
record_lengths = []
for path in ts_files:
    df = pd.read_csv(path, usecols=['date', 'storage'], parse_dates=['date'])
    record_lengths.append(int(df['storage'].notna().sum()))

rl = pd.Series(record_lengths, name='days_with_storage')
print(rl.describe())

fig, ax = plt.subplots(figsize=(8, 4))
rl.plot.hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Days with non-NaN storage')
ax.set_ylabel('Number of reservoirs')
ax.set_title('ResOpsUS — Record length distribution')
plt.tight_layout()
plt.show()

In [ ]:
# --- Example time series for 4 reservoirs ---
sample_ids = [os.path.splitext(os.path.basename(f))[0] for f in ts_files[:4]]

fig, axes = plt.subplots(len(sample_ids), 1, figsize=(12, 3 * len(sample_ids)), sharex=False)

for ax, res_id in zip(axes, sample_ids):
    df = pd.read_csv(
        os.path.join(ts_dir, f'{res_id}.csv'),
        parse_dates=['date'], index_col='date'
    )
    df['storage'].plot(ax=ax, label='Storage (MCM)', color='steelblue')
    if 'outflow' in df.columns and df['outflow'].notna().any():
        df['outflow'].plot(ax=ax, label='Outflow (m³/s)', color='salmon', alpha=0.7)
    ax.set_title(f'Reservoir GRAND_ID = {res_id}')
    ax.set_ylabel('Value')
    ax.legend(loc='upper left')

plt.suptitle('ResOpsUS — Example reservoir time series', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- ResOps metadata: per-variable coverage ---
for var in ['STORAGE', 'OUTFLOW', 'INFLOW', 'ELEVATION', 'EVAPORATION']:
    if var in resops.columns:
        n = int(resops[var].sum())
        print(f"{var:15s}: {n:4d} / {len(resops)} reservoirs")

---
## 3  Summary

| | POINTDATA | ResOpsUS+CARS |
|---|---|---|
| **Type** | Gridded model output (PCR-GLOBWB 2) | Observed reservoir operations |
| **Spatial** | 312×348 grid, ~5 arcmin, CONUS | 676 point reservoirs, CONUS |
| **Temporal** | 1979–2023, daily / monthly / annual | 1975–2020, daily |
| **Key variables** | discharge, storage fraction, inflow, ET, runoff, groundwater, water withdrawal by sector | storage, outflow, elevation, inflow_sim, meteo forcings |
| **Format** | NetCDF3 (most) + NetCDF4 (`10_param/`, `reservoir_use.nc`) | CSV + NetCDF4 + Shapefiles |
| **Count** | 45 `.nc` + 366 parameter files | 676 CSVs, 676 NC4s, 5 attribute CSVs, 2 shapefiles |
| **Note** | `netcdf4` lib needed for `10_param_RF_bounds_final/` and `reservoir_use.nc` | `netcdf4` lib needed for `.nc` time series |